# BINF 4002 — Machine Learning for Biomedical Data
## Homework Assignment: End-to-End ML Across Biomedical Domains

**Total Points: 100** (select any **two** of the three problems for grading; the third is optional bonus worth up to 10 extra-credit points)

| Problem | Domain | Points |
|---------|--------|--------|
| **Problem 1** | Biology — Pan-Cancer Gene Expression Classification | 50 |
| **Problem 2** | Clinical/EHR — Preventive Screening Compliance Prediction | 50 |
| **Problem 3** | Public Health — Influenza Forecasting | 50 |

### Instructions
1. **Choose two** problems to submit for grading.
2. You may complete the third for **up to 10 bonus points**.
3. All code must run **end-to-end in Google Colab**.
4. You are **encouraged** to use GenAI tools (Gemini, ChatGPT, Claude, Copilot) to understand concepts, debug, and implement. However, you must:
   - Understand and be able to explain every line of your code.
   - Write all discussion answers in your own words with genuine analytical depth.
   - **Submit a link to your primary GenAI conversation(s)** used during this assignment (e.g., a shared ChatGPT/Claude conversation link). This is for transparency, not penalization.
6. Submit this notebook as a `.ipynb` file with all cells executed and outputs visible.

### What we expect

Beyond working code and results, we expect you to think out loud throughout your notebook. In the problems below, comment on your model's performance across all relevant axes: not just a single number.

- What concerns would you have about adapting these results to a real-world setting?
- How would you extend your approach given more time or data, and what barriers would you expect?
- What trade-offs exist in your design choices, and why did you resolve them the way you did?

A notebook that achieves modest results but demonstrates genuine understanding will score higher than one with strong numbers and no interpretation.

In [ ]:
# ============================================================
# PASTE YOUR GenAI CONVERSATION LINK(S) HERE
# ============================================================
GENAI_LINKS = [
    # "https://chatgpt.com/share/...",
    # "https://claude.ai/share/...",
]


---
## Environment Setup (Run First)


In [1]:
# ============================================================
# SHARED DEPENDENCIES — run this cell once
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn umap-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print("✅ All shared dependencies loaded.")


✅ All shared dependencies loaded.


---
---
# Problem 1: Pan-Cancer Gene Expression Classification

## Background
RNA sequencing (RNA-Seq) measures the expression levels of thousands of genes simultaneously, producing a molecular "fingerprint" of a tissue sample. Different cancer types originate from different tissues and harbor distinct transcriptomic programs. In this problem, you will build a classifier that predicts cancer type from gene expression profiles, exploring how dimensionality reduction affects performance in a high-dimensional, low-sample-size (p >> n) setting.

## Dataset
The **UCI ML Gene Expression Cancer RNA-Seq** dataset contains 801 samples across 5 cancer types (BRCA, KIRC, LUAD, PRAD, COAD) with 20,531 gene expression features (RSEM normalized counts).

## Learning Objectives
- Handle extreme high-dimensionality (p >> n)
- Compare dimensionality reduction strategies
- Perform stratified cross-validation with class imbalance
- Interpret model outputs in biological context

---


## 1.1 Dataset Loading & Initial Inspection (5 pts)

In [3]:
# ============================================================
# 1.1 LOAD THE DATASET
# ============================================================
# Download the TCGA PANCAN RNA-Seq dataset (801 samples, 20531 genes, 5 cancer types)
# Source: UCI ML Repository — direct archive download

import os, tarfile, glob, shutil

DATA_DIR = '/content/pancan_data'
os.makedirs(DATA_DIR, exist_ok=True)

data_csv = os.path.join(DATA_DIR, 'data.csv')
labels_csv = os.path.join(DATA_DIR, 'labels.csv')

if not os.path.exists(data_csv):
    print("📥 Downloading TCGA PANCAN dataset...")

    # Primary source: UCI archive
    !wget -q --show-progress \
        "https://web.archive.org/web/20250330202645id_/https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz" \
        -O /content/pancan.tar.gz

    # Extract (the tar.gz contains a nested structure)
    print("📦 Extracting...")
    with tarfile.open('/content/pancan.tar.gz', 'r:gz') as outer:
        outer.extractall('/content/pancan_tmp')

    # Handle nested tar files
    for t in glob.glob('/content/pancan_tmp/**/*.tar', recursive=True):
        with tarfile.open(t, 'r') as inner:
            inner.extractall(DATA_DIR)

    # Also move any CSVs found at top level
    for f in glob.glob('/content/pancan_tmp/**/*.csv', recursive=True):
        shutil.copy(f, DATA_DIR)

    # Verify extraction worked
    found = os.listdir(DATA_DIR)
    print(f"  Extracted: {found}")

    if not os.path.exists(data_csv):
        # Try to find the files with any name
        csvs = [f for f in found if f.endswith('.csv')]
        for f in csvs:
            fp = os.path.join(DATA_DIR, f)
            # data file is large (~70MB), labels is small
            size = os.path.getsize(fp) / 1e6
            if size > 10:
                shutil.copy(fp, data_csv)
                print(f"  Mapped {f} ({size:.0f}MB) → data.csv")
            else:
                shutil.copy(fp, labels_csv)
                print(f"  Mapped {f} ({size:.2f}MB) → labels.csv")

    assert os.path.exists(data_csv), (
        "❌ data.csv not found after extraction. "
        "Please download manually from:\n"
        "https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq\n"
        "and upload data.csv + labels.csv to /content/pancan_data/"
    )
    print("✅ Download complete!")
else:
    print("✅ Dataset already downloaded.")

# --- Load ---
X_raw = pd.read_csv(data_csv, index_col=0)
y_series = pd.read_csv(labels_csv, index_col=0)
y_raw = y_series.iloc[:, 0]

print(f"\nFeature matrix shape: {X_raw.shape}")
print(f"Target distribution:\n{y_raw.value_counts()}")
print(f"\nFirst 5 gene columns: {list(X_raw.columns[:5])}")
print(f"Any missing values in X: {X_raw.isnull().any().any()}")
print(f"Any missing values in y: {y_raw.isnull().any()}")


✅ Dataset already downloaded.

Feature matrix shape: (801, 20531)
Target distribution:
Class
BRCA    300
KIRC    146
LUAD    141
PRAD    136
COAD     78
Name: count, dtype: int64

First 5 gene columns: ['gene_0', 'gene_1', 'gene_2', 'gene_3', 'gene_4']
Any missing values in X: False
Any missing values in y: False


#### Conceptual Question 1.1

**Q: This dataset has 20,531 features but only 801 samples. Explain in your own words:**
1. **Why is this "p >> n" setting problematic for standard ML algorithms?**
2. **Name two distinct strategies for addressing this problem** and briefly explain how they differ conceptually.

---


## 1.2 Problem statement (45 pts)


Build a classifier that predicts cancer type from gene expression profiles.

This dataset has far more features than samples. Your solution should demonstrate that you understand why this matters and how it shapes every downstream decision: from preprocessing through model selection to interpretation. We expect you to explore how aggressively you can compress the feature space before classification degrades, compare at least two modeling approaches, and critically assess your results: classification accuracy on this dataset will likely be very high, and part of your job is to reason about whether that reflects genuine diagnostic power or properties of the dataset itself.

We expect executable code, quality figures, and written explanations that connect your technical choices to the biology and the data structure. Results without justification, or justification without results, will receive partial credit.



---
---
# Problem 2: Predicting Preventive Screening Non-Compliance (Synthea)

## Background
Preventive screenings (colonoscopy, mammography, pap test) are evidence-based interventions recommended by the U.S. Preventive Services Task Force (USPSTF). Despite clear guidelines, many patients fall behind on recommended screenings. Health systems want to **proactively identify patients likely to remain non-compliant** so they can target outreach interventions.

In this problem, you will:
1. **Encode clinical guidelines as logic** to determine screening eligibility and compliance status.
2. **Construct a time-sensitive prediction label** (will this patient remain non-compliant at 12 months?).
3. **Build a classifier** using encounter history, demographics, and prior compliance patterns.

## Dataset
**Synthea** — a synthetic patient generator that produces realistic (but artificial) EHR data. We will use a pre-generated 3,000-patient dataset.

## Key Challenge
This problem requires you to think like a clinical informaticist: you must **define the clinical rules** before you can do any ML. The hardest part isn't the model — it's the label construction and feature engineering.

---


## 2.1 Dataset Loading & Initial Inspection (5 pts)

In [2]:
# ============================================================
# 2.1 LOAD SYNTHEA DATA
# ============================================================
# Synthea generates synthetic EHR data. Generating 3,000 patients
# takes ~10+ minutes on Colab, so we provide pre-generated CSVs.

import os

# === Download pre-generated Synthea data ===
SYNTHEA_DRIVE_URL = "142NYfjgHSSaf8yTx3RgOixfl4G3l8sAs"  # Instructor: replace before distributing

csv_dir = '/content/synthea_csv'
os.makedirs(csv_dir, exist_ok=True)

# Check if data already exists
if not os.path.exists(f'{csv_dir}/patients.csv'):
    print("📥 Downloading pre-generated Synthea data...")
    # Instructor will provide a Google Drive folder with the CSVs
    # Students: if the download link doesn't work, generate data with Option B below
    try:
        # Try gdown for Google Drive
        !pip install -q gdown
        import gdown
        gdown.download_folder(
            f"https://drive.google.com/drive/folders/{SYNTHEA_DRIVE_URL}",
            output=csv_dir, quiet=False
        )
        print("✅ Download complete!")
    except Exception as e:
        print(f"⚠️ Google Drive download failed: {e}")
else:
    print("✅ Synthea data already exists.")

# List files
for f in sorted(os.listdir(csv_dir)):
    size_mb = os.path.getsize(f'{csv_dir}/{f}') / 1e6
    print(f"  {f:40s} {size_mb:8.2f} MB")


📥 Downloading pre-generated Synthea data...


Retrieving folder contents


Processing file 14N6z6843HepiHrKBqbkJwQOralaTsXSk conditions.csv
Processing file 1_PifwnZUlitOh4D-3KbSyYRNw-fZ1_gf encounters.csv
Processing file 1aBsLX2puZPrGoAriNRzRgp5Mb-lxZ-vf observations.csv
Processing file 11E_AKxLqeB_rXN1IMI8zqA8Q_xcI1Vz7 patients.csv
Processing file 1RmHSbwDX8tm7v7WipcDuG1SlcSDl8xin procedures.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=14N6z6843HepiHrKBqbkJwQOralaTsXSk
To: /content/synthea_csv/conditions.csv
100%|██████████| 13.7M/13.7M [00:00<00:00, 146MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_PifwnZUlitOh4D-3KbSyYRNw-fZ1_gf
To: /content/synthea_csv/encounters.csv
100%|██████████| 51.9M/51.9M [00:00<00:00, 168MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1aBsLX2puZPrGoAriNRzRgp5Mb-lxZ-vf
From (redirected): https://drive.google.com/uc?id=1aBsLX2puZPrGoAriNRzRgp5Mb-lxZ-vf&confirm=t&uuid=b2e4fcd6-29a2-452e-8f4a-f8b2a16cdb6f
To: /content/synthea_csv/observations.csv
100%|██████████| 307M/307M [00:01<00:00, 169MB/s] 
Downloading...
From: https://drive.google.com/uc?id=11E_AKxLqeB_rXN1IMI8zqA8Q_xcI1Vz7
To: /content/synthea_csv/patients.csv
100%|██████████| 896k/896k [00:00<00:00, 19.3MB/s]
Downloading...
From: https://drive.goo

✅ Download complete!
  conditions.csv                              13.70 MB
  encounters.csv                              51.89 MB
  observations.csv                           306.72 MB
  patients.csv                                 0.90 MB
  procedures.csv                              89.12 MB



Download completed


In [3]:
# ============================================================
# 2.1b LOAD KEY TABLES
# ============================================================

import pandas as pd

patients = pd.read_csv(f'{csv_dir}/patients.csv')
encounters = pd.read_csv(f'{csv_dir}/encounters.csv')
conditions = pd.read_csv(f'{csv_dir}/conditions.csv')
procedures = pd.read_csv(f'{csv_dir}/procedures.csv')
observations = pd.read_csv(f'{csv_dir}/observations.csv')

# Parse dates — Synthea uses START/STOP, not DATE, for most tables
# Strip timezone info to avoid UTC comparison issues
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['DEATHDATE'] = pd.to_datetime(patients['DEATHDATE'], errors='coerce')

# Use utc=True then strip tz for consistent naive timestamps
for df, date_cols in [
    (encounters, ['START', 'STOP']),
    (conditions, ['START', 'STOP']),
    (procedures, ['START', 'STOP']),
    (observations, ['DATE']),
]:
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors='coerce').dt.tz_localize(None)

print(f"Patients:     {patients.shape}")
print(f"Encounters:   {encounters.shape}")
print(f"Conditions:   {conditions.shape}")
print(f"Procedures:   {procedures.shape}")
print(f"Observations: {observations.shape}")

print(f"\nGender distribution:\n{patients['GENDER'].value_counts()}")
print(f"\nAge range: {patients['BIRTHDATE'].min().date()} to {patients['BIRTHDATE'].max().date()}")


Patients:     (3000, 28)
Encounters:   (155309, 15)
Conditions:   (95009, 7)
Procedures:   (422270, 10)
Observations: (1757281, 9)

Gender distribution:
GENDER
F    1563
M    1437
Name: count, dtype: int64

Age range: 1916-04-09 to 2026-04-03


### Conceptual Question 2.1

**Q:** Before writing any ML code, answer:
1. **Look up the current USPSTF guidelines with "A" or "B" recommandations.** For each of the following screenings, state: (a) the recommended **starting age**, (b) the recommended **screening frequency**, and (c) any **eligibility criteria** (sex, conditions):
   - Colorectal cancer screening (colonoscopy)
   - Breast cancer screening (mammography)
   - Cervical Cancer (cervical cytology alone)
   
   You will need this information to build the screening rule configurations in the next section.

2. **What does "non-compliance" mean in this context?** Why is the distinction between "never eligible" and "eligible but non-compliant" critical for correct label construction?

3. **Synthea generates data using clinical modules that often follow guidelines deterministically.** How might this affect the difficulty and realism of your prediction task?

---


### Answer

1. USPSTF A and B recommendations

    **Colorectal cancer** -- screening with colonoscopy is one acceptable strategy amonng several (e.g. FIT, sDNA-FIT)
      - Starting age: 45 years (screening is offered from 45 through 75).
      - Frequency: depends on the test. Colonoscopy every 10 years is the standard (since Synthea dataset mostly encodes colonoscopies)
      - Level of evidence: Grade B for ages 45–49; A for ages 50–75.
      - Eligibility: Adults at "average risk"" of colorectal cancer. applies to men and women. We'll **exclude** high-risk patients/syndromes such as personal history of CRC, IBD, Lynch syndrome, FAP, prior adenomatous polyps, as those outside "average risk" will need different screening recommendations.

    **Breast cancer** -- screening with mammography (Fun fact: In Asian countries like Thailand, ultrasonography is often used as a supplement to mammography because women tend to have denser breast tissue, which can make mammography less sensitive).
      - Starting age: 40 years through 74 years
      - Frequency: Every 2 years (biennial)
      - Level of evidence: Grade B for ages 40-74
      - Eligibility: Cisgender women and other invidiuals assigned female who are at average risk; this appplies to those even with certain risk factors (e.g. family history of breast cancer, dense breasts), but not to very high-risk situations. **Exclusions** include prior breast cancer, BRCA1/BRCA2 carriers, prior chest radiation at young age, high-risk breast lesions on biopsy.

    **Cervical cancer** -- cervical cytology alone (Pap smear)
      - Starting age: 21 years (not before 21). Stopping at 65 (no routine screening after 65)
      - Frequency: Every 3 years with cytology alone
      - Level of evidence: A
      - Eligibility: Persons with a cervix (framed around women in the evidence); not for <21; not routine after 65 with adequate prior negative screening; **Exclusions** include those s/p hysterectomy with cervix removal (for benigh lesions) without a continued indication to screen, prior high-grade cervical lesion or cervical cancer, immunocompromised (HIV), in utero DES exposure

2. What "non-compliance" means here, and never eligible vs eligible but non-compliant
    - A patient is "compliant" with a screening if they have had the recommended test within the recommended interval, given they are someone the guideline applies to
    - Non-compliance usually means that the patient **meet guideline criteria to be screened** (right age/sex/risk and not excluded) but **has not completed** the recommended test within the recommended interval (or at all when due).
    - Never eligible: the patients are outside the guideline denominator (e.g. wrong age, no cervix, already met stop screening criteria). They should not count as "failed to comply" with a screening program, because no screening was indicated.
    - Why it matters! if we label everyone with a recent test as "non-compliant", we mix people who were not supposed to be screened with people who were supposed to be screened but missed care. This dilutes or reverses the causal/clinical meaning of the label, and makes models learn **eligibility** instead of **underscreen among eligible patients**.

3. Synthea and deterministic guideline-following

    Synthetic builds timelines with guideline-style modules, so the outcome we treat as "ground truth" is **NOT INDEPENDENT** of the eligibility and interval rules we use to define compliance in the first place--basically, it is partly the **same logic**. Additionally, non-compliance in Synthea shows up mostly as stochastic or module noise, whereas in real care it is shaped by **social determinants, access, continuity with a usual source of care, and how patients experience the health system**--dimensions Synthea does not model.

    We also expect strong model metrics. And good performance on Synthea will likely overstate how well the model would work on real EHRs. Feature importance from Synthea might not transfer. Variables that rise to the top can be almost circular with the label rather than drivers of underuse in practice. The patient who are hardest for real outreach--for example, people who are otherwise well connected to care but still not screened--are underrepresented relative to real populations.

    So synthea is best used to validate the pipeline, not to prove that the model captures clinically actionable non-compliance. External validation on a real EHR, or at least a more realistic simulator like MIMIC, would be needed before deploying.



## 2.2 Problem statement (45 pts)


Build a classifier that predicts which patients will remain non-compliant with preventive screenings 12 months from now.

This problem requires clinical reasoning before any modeling: you need to translate screening guidelines into computable rules, construct labels that reflect a meaningful clinical question, and engineer features from multiple EHR tables; all while respecting a temporal prediction boundary. The modeling itself is standard once those pieces are right. Be deliberate about how you split the data (the same patient can appear across multiple screening contexts), and about what your model actually learns: including whether its strongest signals would survive contact with real-world EHR data, where non-compliance is driven by factors Synthea cannot simulate.

We expect executable code, figures, and written explanations. We also consider pipeline efficiency (runtime and memory) given the dataset scale.

---
---
# Problem 3: Influenza-Like Illness (ILI) Forecasting

## Background
The CDC's ILINet surveillance system collects weekly reports of influenza-like illness (ILI) from outpatient healthcare providers across the US, organized by HHS Region. Accurate forecasting of ILI rates 1–4 weeks ahead is a critical public health capability that informs hospital staffing, antiviral stockpiling, and public messaging.

This is a **time series forecasting** problem with strong seasonality, making it fundamentally different from the cross-sectional classification in Problems 1 and 2.

## Dataset
CDC FluView ILINet data, available via the CMU Delphi Epidata API or direct CDC download. Weekly %ILI (weighted) by HHS region, spanning 20+ seasons.

## Learning Objectives
- Implement proper **temporal train/test splits** (no random splitting!)
- Build and compare time-series baselines vs. ML approaches
- Evaluate at multiple forecast horizons
- Understand the challenges of epidemic peak prediction

---


## 3.1 Dataset Loading

In [ ]:
# ============================================================
# 3.1 LOAD CDC ILINET DATA
# ============================================================

# Download ILI data from the CMU Delphi Epidata API (reliable, no auth needed)
import requests

ili_raw = None

try:
    url = "https://api.delphi.cmu.edu/epidata/fluview"
    params = {
        "regions": "nat",
        "epiweeks": "201001-202352"
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    if data["result"] != 1:
        raise ValueError(f"API returned result={data['result']}")

    ili_raw = pd.DataFrame(data["epidata"])

    # Standardize columns
    rename_map = {}
    if 'wili' in ili_raw.columns:
        rename_map['wili'] = 'weighted_ili'
    elif 'ili' in ili_raw.columns and 'weighted_ili' not in ili_raw.columns:
        rename_map['ili'] = 'weighted_ili'
    ili_raw = ili_raw.rename(columns=rename_map)

    # Derive year/week from epiweek
    if 'epiweek' in ili_raw.columns:
        ili_raw['epiweek'] = ili_raw['epiweek'].astype(int)
        ili_raw['year'] = ili_raw['epiweek'] // 100
        ili_raw['week'] = ili_raw['epiweek'] % 100

    # Derive week_start date
    if 'year' in ili_raw.columns and 'week' in ili_raw.columns:
        ili_raw['week_start'] = pd.to_datetime(
            ili_raw['year'].astype(str) + '-W' +
            ili_raw['week'].astype(str).str.zfill(2) + '-1',
            format='%G-W%V-%u', errors='coerce'
        )

    print(f"✅ Loaded ILI data from Delphi Epidata API")

except Exception as e:
    print(f"⚠️ Delphi API failed: {e}")

print(f"\nDataset shape: {ili_raw.shape}")
print(f"Columns: {list(ili_raw.columns)}")
print(f"Year range: {ili_raw['year'].min()} to {ili_raw['year'].max()}")
print(f"Regions: {ili_raw['region'].nunique()}")
print(ili_raw.head())


✅ Loaded ILI data from Delphi Epidata API

Dataset shape: (730, 19)
Columns: ['release_date', 'region', 'issue', 'epiweek', 'lag', 'num_ili', 'num_patients', 'num_providers', 'num_age_0', 'num_age_1', 'num_age_2', 'num_age_3', 'num_age_4', 'num_age_5', 'weighted_ili', 'ili', 'year', 'week', 'week_start']
Year range: 2010 to 2023
Regions: 1
  release_date region   issue  epiweek  lag  num_ili  num_patients  \
0   2013-12-31    nat  201352   201001  207    14299        721138   
1   2013-12-31    nat  201352   201002  206    14088        770895   
2   2013-12-31    nat  201352   201003  205    14757        766177   
3   2013-12-31    nat  201352   201004  204    15122        785580   
4   2013-12-31    nat  201352   201005  203    16037        767773   

   num_providers  num_age_0  num_age_1 num_age_2  num_age_3  num_age_4  \
0           1996       4998       3961      None       3333       1244   
1           2016       4877       4614      None       2793       1182   
2           205

In [ ]:
# ============================================================
# 3.1b PICK A SINGLE REGION FOR FOCUSED ANALYSIS
# ============================================================
# We'll focus on one region

TARGET_REGION = ili_raw['region'].unique()[0]  # First region
print(f"Focusing on: {TARGET_REGION}")

ili = ili_raw[ili_raw['region'] == TARGET_REGION].copy()

# Create a proper date index
if 'week_start' in ili.columns:
    ili = ili.sort_values('week_start').reset_index(drop=True)
    ili['date'] = ili['week_start']
else:
    # Construct date from year + week
    ili['date'] = pd.to_datetime(ili['year'].astype(str) + '-W' +
                                  ili['week'].astype(str).str.zfill(2) + '-1',
                                  format='%Y-W%W-%w', errors='coerce')
    ili = ili.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

print(f"Records for {TARGET_REGION}: {len(ili)}")
print(f"Date range: {ili['date'].min()} to {ili['date'].max()}")
print(f"\n%ILI summary stats:")
print(ili['weighted_ili'].describe())


Focusing on: nat
Records for nat: 730
Date range: 2010-01-04 00:00:00 to 2023-12-25 00:00:00

%ILI summary stats:
count    730.000000
mean       2.017086
std        1.353939
min        0.529813
25%        1.108860
50%        1.556255
75%        2.370798
max        7.521330
Name: weighted_ili, dtype: float64


## 3.2 Problem statement (50 pts)

Build a forecasting model that predicts ILI rates 1, 2, 3, and 4 weeks ahead.

Time series data requires fundamentally different treatment than cross-sectional data: your data splitting, feature engineering, baseline selection, and evaluation must all reflect this. A strong submission will establish baselines that are genuinely hard to beat (not just strawmen), honestly assess where ML adds value over simpler strategies, and characterize when forecasts fail, not just how much error they have on average. Consider how your approach compares to the state of the art (the CDC runs a public forecasting challenge FluSight), what exogenous data sources could improve predictions beyond lagged ILI values, and whether a single temporal split gives you enough confidence in your performance estimates.

We expect executable code, figures, and written explanations.